In [10]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

In [11]:
embed = nn.Embedding(10, 3)

In [12]:
df = pd.read_csv("./data/ml-latest-small/ratings.csv")
df = df.drop(columns=["timestamp"])

# Encode user and movie IDs to contiguous 0-based indices
user2idx = {u: i for i, u in enumerate(df["userId"].unique())}
movie2idx = {m: i for i, m in enumerate(df["movieId"].unique())}

df["user"] = df["userId"].map(user2idx)
df["movie"] = df["movieId"].map(movie2idx)

n_users = df["user"].nunique()
n_movies = df["movie"].nunique()
print(f"Users: {n_users}, Movies: {n_movies}, Ratings: {len(df)}")
print(f"Sparsity: {1 - len(df) / (n_users * n_movies):.2%}")
df.head()


Users: 610, Movies: 9724, Ratings: 100836
Sparsity: 98.30%


,userId,movieId,rating,user,movie
0,1,1,4.0,0,0
1,1,3,4.0,0,1
2,1,6,4.0,0,2
3,1,47,5.0,0,3
4,1,50,5.0,0,4


In [13]:
class RatingsDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor(df["user"].values, dtype=torch.long)
        self.movies = torch.tensor(df["movie"].values, dtype=torch.long)
        self.ratings = torch.tensor(df["rating"].values, dtype=torch.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return self.users[idx], self.movies[idx], self.ratings[idx]


train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

train_loader = DataLoader(RatingsDataset(train_df), batch_size=1024, shuffle=True)
test_loader = DataLoader(RatingsDataset(test_df), batch_size=1024, shuffle=False)
print(f"Train: {len(train_df):,}  Test: {len(test_df):,}")


Train: 80,668  Test: 20,168


In [14]:
embedding = nn.Embedding(1, 1, max_norm=1.0)


In [15]:
class MF_MLP(nn.Module):
    def __init__(self, n_users: int, n_movies: int, emb_dim: int = 64):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_movies, emb_dim)
        self.user_emb.weight.data.uniform_(0, 0.05)
        self.item_emb.weight.data.uniform_(0, 0.05)

    def forward(
        self, user_ids: torch.LongTensor, movie_ids: torch.LongTensor
    ) -> torch.Tensor:
        user_embeddings = self.user_emb(user_ids)
        item_embeddings = self.item_emb(movie_ids)
        return (user_embeddings * item_embeddings).sum(dim=1)


model = MF_MLP(n_users, n_movies, emb_dim=64)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")


MF_MLP(
  (user_emb): Embedding(610, 64)
  (item_emb): Embedding(9724, 64)
)
Parameters: 661,376


In [16]:
num_epochs = 20
for epoch in range(1, num_epochs + 1):
    model.train()
    total_loss = 0
    for (
        users_current_batch,
        movies_current_batch,
        ratings_current_batch,
    ) in train_loader:
        optimizer.zero_grad()
        preds = model(users_current_batch, movies_current_batch)
        loss = loss_fn(preds, ratings_current_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(ratings_current_batch)
    train_rmse = (total_loss / len(train_loader.dataset)) ** 0.5
    print(f"Epoch {epoch:2d}/{num_epochs}  train RMSE: {train_rmse:.4f}")


Epoch  1/20  train RMSE: 3.4945
Epoch  2/20  train RMSE: 2.9070
Epoch  3/20  train RMSE: 2.0411
Epoch  4/20  train RMSE: 1.4796
Epoch  5/20  train RMSE: 1.2533
Epoch  6/20  train RMSE: 1.1325
Epoch  7/20  train RMSE: 1.0534
Epoch  8/20  train RMSE: 0.9971
Epoch  9/20  train RMSE: 0.9555
Epoch 10/20  train RMSE: 0.9241
Epoch 11/20  train RMSE: 0.8996
Epoch 12/20  train RMSE: 0.8804
Epoch 13/20  train RMSE: 0.8650
Epoch 14/20  train RMSE: 0.8526
Epoch 15/20  train RMSE: 0.8425
Epoch 16/20  train RMSE: 0.8345
Epoch 17/20  train RMSE: 0.8277
Epoch 18/20  train RMSE: 0.8220
Epoch 19/20  train RMSE: 0.8176
Epoch 20/20  train RMSE: 0.8136


In [17]:
model.eval()
total_loss = 0
with torch.no_grad():
    for users, movies, ratings in test_loader:
        preds = model(users, movies)
        total_loss += loss_fn(preds, ratings).item() * len(ratings)
test_rmse = (total_loss / len(test_loader.dataset)) ** 0.5
print(f"Test RMSE: {test_rmse:.4f}")


Test RMSE: 1.1119


In [18]:
# sample predictions vs ground truth
sample = test_df.sample(10, random_state=0)
users = torch.tensor(sample["user"].values, dtype=torch.long)
movies = torch.tensor(sample["movie"].values, dtype=torch.long)
with torch.no_grad():
    preds = model(users, movies).numpy()

print(f"\n{'Actual':>8}  {'Predicted':>9}")
for actual, pred in zip(sample["rating"].values, preds):
    print(f"{actual:8.1f}  {pred:9.4f}")



  Actual  Predicted
     4.5     4.4788
     2.5     2.7638
     3.5     3.8516
     1.0     2.5839
     2.0     3.6407
     4.0     3.8895
     4.0     4.1825
     5.0     4.2823
     2.0     2.8729
     3.5     3.7648
